# 🧠 NEXUS AI — QLoRA Fine-Tuning
**Model**: Llama-3.2-3B (or Mistral-7B) | **Method**: QLoRA via Unsloth

**Steps**:
1. Install Unsloth + deps
2. Load base model in 4-bit
3. Apply LoRA adapters
4. Load NEXUS training data
5. Train with SFT
6. Evaluate before/after
7. Export to GGUF → serve with Ollama


In [ ]:
# ── Cell 1: Install ──────────────────────────────────────────
%%capture
!pip install unsloth
!pip install --upgrade trl peft accelerate bitsandbytes datasets evaluate rouge-score
print('✅ Installation complete')

In [ ]:
# ── Cell 2: Config ───────────────────────────────────────────
CONFIG = {
    'model_name':   'unsloth/Llama-3.2-3B-Instruct',  # swap to 'mistralai/Mistral-7B-v0.1' for larger
    'max_seq_length': 2048,
    'load_in_4bit': True,
    # LoRA hyperparameters
    'lora_r': 16,          # rank — higher = more parameters adapted
    'lora_alpha': 32,      # scaling
    'lora_dropout': 0.05,
    'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    # Training
    'num_train_epochs': 3,
    'per_device_train_batch_size': 4,
    'gradient_accumulation_steps': 4,     # effective batch = 16
    'learning_rate': 2e-4,
    'warmup_steps': 50,
    'lr_scheduler_type': 'cosine',
    'output_dir': '/content/nexus-finetuned',
    'train_data': '/content/nexus_train.jsonl',   # upload your exported JSONL here
}
print('Config ready')

In [ ]:
# ── Cell 3: Load model ───────────────────────────────────────
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG['model_name'],
    max_seq_length=CONFIG['max_seq_length'],
    load_in_4bit=CONFIG['load_in_4bit'],
    dtype=None,   # auto-detect
)
print(f'✅ Base model loaded: {CONFIG["model_name"]}')

In [ ]:
# ── Cell 4: Apply LoRA ───────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG['lora_r'],
    target_modules=CONFIG['target_modules'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA applied: {trainable/1e6:.1f}M / {total/1e6:.1f}M params trainable ({100*trainable/total:.1f}%)')

In [ ]:
# ── Cell 5: Load + format dataset ────────────────────────────
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template='llama-3')

ALPACA_PROMPT = """Below is an instruction that describes a task. Write a response.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}<|end_of_text|>"""

dataset = load_dataset('json', data_files=CONFIG['train_data'], split='train')

def format_sample(examples):
    return {'text': [
        ALPACA_PROMPT.format(instruction=i, input=inp, output=out)
        for i, inp, out in zip(examples['instruction'], examples['input'], examples['output'])
    ]}

dataset = dataset.map(format_sample, batched=True)
print(f'✅ Dataset loaded: {len(dataset)} samples')
print('Sample prompt (truncated):')
print(dataset[0]['text'][:400])

In [ ]:
# ── Cell 6: Train ────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=CONFIG['max_seq_length'],
    dataset_num_proc=2,
    packing=True,    # pack short sequences for efficiency
    args=TrainingArguments(
        per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
        gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
        warmup_steps=CONFIG['warmup_steps'],
        num_train_epochs=CONFIG['num_train_epochs'],
        learning_rate=CONFIG['learning_rate'],
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim='adamw_8bit',
        lr_scheduler_type=CONFIG['lr_scheduler_type'],
        output_dir=CONFIG['output_dir'],
        save_strategy='epoch',
        report_to='none',    # set 'wandb' if you have W&B
    ),
)

print('🚀 Starting training...')
trainer_stats = trainer.train()
print(f'✅ Training complete! Runtime: {trainer_stats.metrics["train_runtime"]:.0f}s')

In [ ]:
# ── Cell 7: Before/After eval ────────────────────────────────
FastLanguageModel.for_inference(model)

TEST_TICKET = '''Category: database
Priority: high
Title: PostgreSQL cannot connect — too many connections
Description: Application throwing FATAL: remaining connection slots are reserved for non-replication superuser connections'''

prompt = ALPACA_PROMPT.format(
    instruction='You are NEXUS AI, an IT Service Management assistant. Analyse the ticket and provide root cause, resolution steps, and prevention.',
    input=TEST_TICKET,
    output=''
).removesuffix('<|end_of_text|>')

inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print('=== Fine-tuned Model Response ===')
print(response)

In [ ]:
# ── Cell 8: Export to GGUF for Ollama ────────────────────────
# Q4_K_M = good balance of size vs quality
model.save_pretrained_gguf(
    '/content/nexus-gguf',
    tokenizer,
    quantization_method='q4_k_m'
)
print('✅ GGUF model saved to /content/nexus-gguf')
print('Download nexus-finetuned-Q4_K_M.gguf and follow deploy_ollama.sh to serve locally')

In [ ]:
# ── Cell 9: Download from Colab ──────────────────────────────
from google.colab import files
import glob

gguf_files = glob.glob('/content/nexus-gguf/*.gguf')
print(f'Found: {gguf_files}')
# files.download(gguf_files[0])   # uncomment to download
print('Uncomment the last line to download the GGUF file')